## LangGraph ReAct Agent with Tools
Learning Objectives:
- Create tools (functions) that LLMs can call
- Implement the ReAct pattern (Reasoning + Acting)
- Build an agent that decides when to use tools

#### 
Real-World Tools:
-----------------
- Database queries
- API calls (weather, stock prices, etc.)
- File operations
- Web searches
- Send emails/notifications
- Execute code
- Image generation
- Data analysis

### Agent Patterns
- Chain of Thoughts (CoT)
- Tree of Thoughts (ToT)
- ReAct

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()

/Users/sathish/Documents/akila/ML-learning-materials/ai-ml-akila/ml311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatOpenAI(temperature=0) #  do experiment with different temperatures


In [3]:
llm.invoke('Hi my wonderful akila, give me a different response')

AIMessage(content='Hello there, my fabulous friend!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 18, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d90a8-855b-7b80-aaa8-05ecd9c98bd5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 7, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
import my_tools
# examples 
my_tools.get_current_datetime.invoke({'date': 'today date'})

****************************************************************************************************
========> [TOOL] get_current_datetime () -> '2026-04-15 18:20:58'


'2026-04-15 18:20:58'

In [5]:
eval('2+2*1.4/23-34')

-31.878260869565217

In [6]:
all_tools=[my_tools.calculate,my_tools.get_weather, my_tools.get_current_datetime]

from typing_extensions import TypedDict, Annotated
import operator
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

In [7]:

def agent_node(state: AgentState):

    llm_with_tools = llm.bind_tools(all_tools)

    messages = state['messages']

    response = llm_with_tools.invoke(messages)
    print(response)

    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")


    return {'messages': [response]}

In [8]:
state = {"messages": [HumanMessage("today date?")]}
result = agent_node(state)

content='' additional_kwargs={'tool_calls': [{'id': 'call_CmX9mvRYG8D4kprMSu3RQhxC', 'function': {'arguments': '{"date":"today"}', 'name': 'get_current_datetime'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 401, 'total_tokens': 416, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d90a8-8cd4-7c10-b5e4-d9bf3eceffca-0' tool_calls=[{'name': 'get_current_datetime', 'args': {'date': 'today'}, 'id': 'call_CmX9mvRYG8D4kprMSu3RQhxC', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 401, 'output_tokens': 15, 'total_tokens': 416, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0